# Document Structure Diagnostic Notebook

This notebook is built around the project's actual architecture.

**Goal:** understand how a heterogeneous PDF is structured so that expensive downstream extraction (LLM / NER / targeted OCR) can be applied only to useful pages or sections.

```text
PDF
├── bookmarks / outline
└── PyMuPDF page analysis
    ├── raw text
    ├── text blocks / typography
    ├── tables
    └── scan/image evidence
             ↓
       TOC detection
             ↓
       title candidates
             ↓
       TOC ↔ title matching
             ↓
       boundary detection
             ↓
       hierarchy + page ranges
             ↓
          validation
             ↓
       useful-page routing
             ↓
        LLM / NER / OCR
```

The notebook is **diagnostic-first**: we inspect the evidence behind decisions, not just the final score.


## 1. Configuration

Set the PDF you want to investigate. The notebook uses **PyMuPDF** and the project's `src.document_structure` modules.


In [1]:
from pathlib import Path
import sys
import inspect
import pandas as pd
import pymupdf as fitz

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PDF_PATH = PROJECT_ROOT / "data" / "raw" / "MYCOM Operating and Maintenance Manual Refrigeration Unit (1).pdf"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PDF_PATH:", PDF_PATH)


PROJECT_ROOT: c:\Users\intel\Downloads\industrial-equipment-data-extraction
PDF_PATH: c:\Users\intel\Downloads\industrial-equipment-data-extraction\data\raw\MYCOM Operating and Maintenance Manual Refrigeration Unit (1).pdf


## 2. Open PDF and inspect native metadata


In [2]:
doc = fitz.open(PDF_PATH)

print("PDF pages:", len(doc))
print("\nMetadata:")
for key, value in doc.metadata.items():
    print(f"  {key}: {value}")


PDF pages: 1068

Metadata:
  format: PDF 1.5
  title: untitled
  author: mara.riga
  subject: 
  keywords: 
  creator: Adobe Acrobat 10.1.6
  producer: macOS Version 14.4.1 (assemblage 23E224) Quartz PDFContext, AppendMode 1.1
  creationDate: D:20140616145557Z00'00'
  modDate: D:20260701105902Z00'00'
  trapped: 
  encryption: None


## 3. Inspect PDF bookmarks / outline

Bookmarks are an independent structural signal. They are useful when present, but they are not automatically ground truth.


In [3]:
outline = doc.get_toc(simple=False)

print("Bookmark entries:", len(outline))
for item in outline[:100]:
    level, title, page = item[:3]
    print(f"L{level:<2} PDF page {page:<4} {title}")


Bookmark entries: 0


## 4. Run the actual `PageAnalyzer`

The page layer must preserve **raw text AND tables**. This is essential for table-based TOCs whose section/title/page relationships can be destroyed by flattening cells into one text stream.


In [4]:
from src.document_structure.page_analysis import PageAnalyzer

page_analyzer = PageAnalyzer()
pages_meta = page_analyzer.extract_document(doc)

print("Pages analyzed:", len(pages_meta))
print("Page object:", type(pages_meta[0]).__name__ if pages_meta else None)

Consider using the pymupdf_layout package for a greatly improved page layout analysis.
Pages analyzed: 1068
Page object: PageRepresentation


## 5. Inspect the page representation


In [5]:
def get_attr(obj, *names, default=None):
    for name in names:
        if hasattr(obj, name):
            return getattr(obj, name)
    return default

if pages_meta:
    page = pages_meta[0]
    print(type(page).__name__)
    if hasattr(page, "__dict__"):
        for key, value in vars(page).items():
            print(f"{key}: {type(value).__name__}")


PageRepresentation
page_number: int
raw_text: str
text_blocks: list
tables: list
image_count: int
is_scanned: bool
width: float
height: float


## 6. Build extraction inventory

This separates an **extraction problem** from a **TOC detection problem**.


In [6]:
rows = []
for pdf_page, page in enumerate(pages_meta, start=1):
    text = get_attr(page, "raw_text", "text", default="") or ""
    blocks = get_attr(page, "text_blocks", "blocks", default=[]) or []
    tables = get_attr(page, "tables", "extracted_tables", default=[]) or []
    rows.append({
        "pdf_page": pdf_page,
        "text_chars": len(text),
        "text_blocks": len(blocks),
        "table_count": len(tables),
        "has_text": bool(text.strip()),
        "has_table": bool(tables),
    })

page_inventory = pd.DataFrame(rows)
display(page_inventory.head(25))
print("Pages with tables:", int(page_inventory["has_table"].sum()))
print("Pages with no extracted text:", int((~page_inventory["has_text"]).sum()))


,pdf_page,text_chars,text_blocks,table_count,has_text,has_table
0,1,490,14,1,True,True
1,2,364,14,2,True,True
2,3,338,10,1,True,True
3,4,309,8,1,True,True
4,5,2609,101,1,True,True
5,6,1251,45,1,True,True
6,7,310,8,1,True,True
7,8,0,0,0,False,False
8,9,367,11,1,True,True
9,10,321,9,1,True,True


Pages with tables: 520
Pages with no extracted text: 167


## 7. Inspect the known TOC pages

Start with the pages discussed during development: **6, 7 and 187**.

They are 1-based PDF page numbers. Inspect raw text and tables separately.


In [ ]:
FOCUS_PAGES = [6, 7, 187]

for page_number in FOCUS_PAGES:
    if not 1 <= page_number <= len(pages_meta):
        print("Skipping invalid page:", page_number)
        continue

    page = pages_meta[page_number - 1]
    text = get_attr(page, "raw_text", "text", default="") or ""
    tables = get_attr(page, "tables", "extracted_tables", default=[]) or []

    print("\n")
    print(f"PDF PAGE {page_number}")
    print("\n--- RAW TEXT ---")
    print(text[:8000])
    print(f"\n--- TABLES ({len(tables)}) ---")
    for i, table in enumerate(tables, start=1):
        print(f"\nTABLE {i}:")
        print(table)



PDF PAGE 6

--- RAW TEXT ---
PROJECT:
AMMONIASTORAGEREFRIGERATION&TRANSFERSYSTEM/AMMONIACOMPRESSOR
VENDORNAME:
MayekawaMycom
P.O.No.
2012123PO3563
AREA:
455A
FINAL DOCUMENTATION
OPERATING AND MAINTENANCE DATA: TECHNICAL DOCUMENTS
Description
Doc. No.
5.4
5.5
5.6
5.7
5.8
5.9
6.
PIPING
6.1
6.2
6.3
7.
7.1
7.2
7.3
7.4
7.5
7.6
7.7
7.8
7.9
7.10
7.11
LOCALGAUGEBOARD&JBLAYOUT
P1-REF-2012-123-018
CABLESLIST
P1-REF-LST-12-123-004
INSTRUMENTSANDCABLETRAYLAYOUT
P1-REF-2012-123-004
P1-REF-LST-12-123-006
PACKAGEI/OLIST
CONTROLSYSTEMBLOCKDIAGRAM
P1-REF-2012-123-017
INSTRUMENTATION&CONTROL
INSTRUMENTLIST&SETTING
P1-REF-LST-12-123-005
CAUSE&EFFECTSCHART
P1-REF-LST-12-123-011
PUMPMOTORSDRAWING
P1-REF-2012-123-025
P1-REF-2012-123-031
MAINMOTORTERMINALBOXES
P1-REF-2012-123-030
FANMOTORSDRAWING
P1-REF-2012-123-024
P1-REF-DSS-12-123-020
P1-REF-SPC-12-123-001
PIPINGCLASS
ELECTRICALHOOKUP
INSTRUMENTHOOKUP
P1-REF-2012-123-028
SAFETYVALVESCALCULATIONSHEETS
P

## 8. Table-vs-text extraction experiment

Hypothesis:

```text
raw text only
    ↓
"1 Introduction 3 1.1 Scope 4 ..."
```

can lose column relationships, while table extraction can preserve:

```text
section | title | printed page
```

This is why tables are first-class input to TOC analysis.


In [8]:
from src.document_structure.toc import TOCDetector
detector = TOCDetector()
result = detector.explain_page(pages_meta[5])  # page 6, 0-indexed
print(result["is_toc"], result["confidence"])
print(result["positive_evidence"])

True 0.5
{'strong_toc_keywords': [], 'document_register_header': True, 'section_structure': {'match_count': 0, 'ratio': 0.0}, 'page_references': {'match_count': 0, 'ratio': 0.0}, 'dot_leaders': {'match_count': 0, 'ratio': 0.0}, 'toc_tables': [{'rows': 26, 'columns': 3, 'bbox': (47.15196990966797, 138.03709411621094, 547.2897033691406, 458.71401596069336), 'extraction_method': 'pymupdf', 'sample_rows': [['', 'Description', 'Doc. No.'], ['5.4', 'MAIN\x02MOTOR\x02TERMINAL\x02BOXES', 'P1-REF-2012-123-030'], ['5.5', 'MAIN\x02MOTOR\x02TESTING\x02PROCEDURE', 'P1-REF-PRD-12-123-006'], ['5.6', 'FAN\x02MOTORS\x02DRAWING', 'P1-REF-2012-123-024'], ['5.7', 'FAN\x02MOTORS\x02DATA\x02SHEET\x02&\x02CURVES', 'P1-REF-DSS-12-123-019']]}], 'toc_table_count': 1, 'entry_like_structure': {'line_count': 76, 'short_line_ratio': 1.0}, 'layout_consistency': {'consistent_entry_structure': False, 'page_reference_ratio': 0.0}}


## 9. Run the actual `TOCDetector`

The detector should combine multiple signals.

**Positive evidence** may include section numbering, page-reference patterns, TOC wording, dot leaders/alignment, table structure and entry density.

**Negative evidence** may include index-like structure, ordinary tables, prose dominance, or weak section/page-reference patterns.

This matters because an `Index` page is not necessarily useful for reconstructing document hierarchy, while a real TOC may have a non-obvious title such as `Final Documentation`.


In [9]:
from src.document_structure.toc import TOCDetector

toc_detector = TOCDetector(toc_score_threshold=0.40)
detected_tocs = toc_detector.detect_printed_tocs(pages_meta)

print("Detected TOC regions:", len(detected_tocs))
for i, region in enumerate(detected_tocs, 1):
    print(f"\n--- TOC REGION {i} ---")
    print("type:", region.toc_type)
    print("pages:", region.page_numbers)
    print("confidence:", round(region.confidence, 3))
    print("entries:", len(region.entries))


Detected TOC regions: 18

--- TOC REGION 1 ---
type: printed
pages: [5, 6]
confidence: 0.5
entries: 69

--- TOC REGION 2 ---
type: printed
pages: [188]
confidence: 0.5
entries: 36

--- TOC REGION 3 ---
type: printed
pages: [282, 283, 284, 285]
confidence: 0.43
entries: 20

--- TOC REGION 4 ---
type: printed
pages: [409]
confidence: 0.63
entries: 77

--- TOC REGION 5 ---
type: printed
pages: [549]
confidence: 0.5
entries: 2

--- TOC REGION 6 ---
type: printed
pages: [554]
confidence: 0.5
entries: 3

--- TOC REGION 7 ---
type: printed
pages: [599]
confidence: 0.5
entries: 2

--- TOC REGION 8 ---
type: printed
pages: [601]
confidence: 0.474
entries: 6

--- TOC REGION 9 ---
type: printed
pages: [603]
confidence: 0.5
entries: 3

--- TOC REGION 10 ---
type: printed
pages: [635, 636]
confidence: 0.5
entries: 21

--- TOC REGION 11 ---
type: printed
pages: [751]
confidence: 0.5
entries: 2

--- TOC REGION 12 ---
type: printed
pages: [753]
confidence: 0.474
entries: 6

--- TOC REGION 13 ---
type:

## 10. Inspect TOC regions and their entries


In [ ]:
def dump_object(obj):
    if hasattr(obj, "model_dump"):
        data = obj.model_dump()
    elif hasattr(obj, "__dict__"):
        data = vars(obj)
    else:
        print(repr(obj))
        return
    for key, value in data.items():
        print(f"{key}: {value}")

for i, region in enumerate(detected_tocs, 1):
    print()
    print(f"TOC REGION {i}")
    dump_object(region)

    print("\nFirst entries:")
    for entry in region.entries[:30]:
        print(
            f"[{getattr(entry,'section_number',None) or '•':>10}] "
            f"L{getattr(entry,'level',None)} "
            f"{str(getattr(entry,'text',''))[:90]:<90} "
            f"printed_ref={getattr(entry,'printed_page_ref',None)}"
        )



TOC REGION 1
pages: [5, 6]
entries: [TOCEntry(text='DOCUMENTS\x02&\x02DRAWINGS\x02LIST', section_number='1.1', level=2, printed_page_ref='P1-REF-LST-12-123-007', reference_kind='doc_code', source_page=5, confidence=0.8), TOCEntry(text='SUBVENDOR\x02LIST', section_number='1.2', level=2, printed_page_ref='P1-REF-LST-12-123-003', reference_kind='doc_code', source_page=5, confidence=0.8), TOCEntry(text='NOISE\x02DATA\x02SHEET', section_number='1.3', level=2, printed_page_ref='P1-REF-DSS-12-123-008', reference_kind='doc_code', source_page=5, confidence=0.8), TOCEntry(text='LUBRICANT\x02LIST', section_number='1.4', level=2, printed_page_ref='P1-REF-SPC-12-123-002', reference_kind='doc_code', source_page=5, confidence=0.8), TOCEntry(text='PAINTING\x02SPECIFICATIONS', section_number='1.5', level=2, printed_page_ref='P1-REF-SPC-12-123-003', reference_kind='doc_code', source_page=5, confidence=0.8), TOCEntry(text='PACKAGE\x02DRAWINGS', section_number='2', level=1, printed_page_ref=None, referen

## 11. Inspect page-level TOC reasoning

The objective is to make the detector explain itself.

For interesting pages we want to see:

```text
positive evidence
negative evidence
extraction source
score / confidence
final decision
```

Use the actual public API exposed by the installed `TOCDetector`.


In [11]:
print("TOCDetector public methods:")
for name in dir(toc_detector):
    if not name.startswith("_"):
        print(" ", name)


TOCDetector public methods:
  analyze_page
  detect_printed_tocs
  explain_page
  toc_score_threshold


In [ ]:
if hasattr(toc_detector, "analyze_page"):
    for page_number in FOCUS_PAGES:
        if 1 <= page_number <= len(pages_meta):
            result = toc_detector.analyze_page(pages_meta[page_number - 1])
            print()
            print(f"PAGE {page_number} TOC ANALYSIS")
            dump_object(result)
else:
    print("No public analyze_page() method in this version.")
    print("Inspect the returned TOCRegion objects from detect_printed_tocs().")



PAGE 6 TOC ANALYSIS
page_number: 6
is_toc: True
confidence: 0.5
evidence: {'positive': {'strong_toc_keywords': [], 'document_register_header': True, 'section_structure': {'match_count': 0, 'ratio': 0.0}, 'page_references': {'match_count': 0, 'ratio': 0.0}, 'dot_leaders': {'match_count': 0, 'ratio': 0.0}, 'toc_tables': [{'rows': 26, 'columns': 3, 'bbox': (47.15196990966797, 138.03709411621094, 547.2897033691406, 458.71401596069336), 'extraction_method': 'pymupdf', 'sample_rows': [['', 'Description', 'Doc. No.'], ['5.4', 'MAIN\x02MOTOR\x02TERMINAL\x02BOXES', 'P1-REF-2012-123-030'], ['5.5', 'MAIN\x02MOTOR\x02TESTING\x02PROCEDURE', 'P1-REF-PRD-12-123-006'], ['5.6', 'FAN\x02MOTORS\x02DRAWING', 'P1-REF-2012-123-024'], ['5.7', 'FAN\x02MOTORS\x02DATA\x02SHEET\x02&\x02CURVES', 'P1-REF-DSS-12-123-019']]}], 'toc_table_count': 1, 'entry_like_structure': {'line_count': 76, 'short_line_ratio': 1.0}, 'layout_consistency': {'consistent_entry_structure': False, 'page_reference_ratio': 0.0}}, 'negative

## 12. Expected reasoning output

A useful diagnostic should be interpretable approximately as:

```text
PAGE 187

TOC = TRUE
confidence = 0.84

POSITIVE
  + TOC-like table
  + repeated section numbering
  + many page references
  + aligned title/reference columns

NEGATIVE
  - title is not an explicit TOC keyword
  - weak dot-leader evidence

EXTRACTION
  source = table
```

The important part is **evidence**, not the exact numeric score.


## 13. Negative cases: Index and ordinary tables

Explicitly test pages that look TOC-like but should not become TOCs.

Examples:

- `Index`;
- figure/table lists;
- ordinary engineering data tables;
- pages with many page references but no hierarchy;
- warning/notice pages.

This protects against the simplistic rule:

```text
many page references = TOC
```


In [ ]:
NEGATIVE_CASE_PAGES = []

for page_number in NEGATIVE_CASE_PAGES:
    if 1 <= page_number <= len(pages_meta):
        page = pages_meta[page_number - 1]
        text = get_attr(page, "raw_text", "text", default="") or ""
        tables = get_attr(page, "tables", "extracted_tables", default=[]) or []
        print()
        print(f"NEGATIVE CASE PAGE {page_number}")
        print(text[:5000])
        print("tables:", len(tables))


## 14. Title candidates and structure analyzer

TOC detection and title detection are separate.

Title candidates should combine typography/layout with semantic and structural evidence. A large font alone is not sufficient because covers, labels and other artifacts can also be large.


In [14]:
from src.document_structure.analyzer import DocumentSegmenter

analyzer = DocumentSegmenter()

print("DocumentSegmenter public methods:")
for name in dir(analyzer):
    if not name.startswith("_"):
        try:
            member = getattr(analyzer, name)
            if callable(member):
                print(f"  {name}{inspect.signature(member)}")
        except (TypeError, ValueError):
            print(" ", name)


DocumentSegmenter public methods:
  segment(pages: 'list[PageRepresentation]', toc_analyses: 'list[TOCAnalysis] | None' = None, title_candidates: 'dict[int, list[TitleCandidate]] | None' = None) -> 'list[DocumentSegment]'


## 15. Run the actual orchestration API

Do not guess the method signature. Inspect it above, then set the exact method name used by the current source.


In [15]:
STRUCTURE_METHOD_NAME = "segment"

if STRUCTURE_METHOD_NAME is not None:
    structure_method = getattr(analyzer, STRUCTURE_METHOD_NAME)
    print("Method:", STRUCTURE_METHOD_NAME)
    print("Signature:", inspect.signature(structure_method))

    # DocumentSegmenter.segment() wants the per-page TOCAnalysis list
    # (not the grouped TOCRegion list from section 9) -- recompute it
    # here since detect_printed_tocs() doesn't expose it directly.
    toc_analyses = [toc_detector.analyze_page(page) for page in pages_meta]

    segments = analyzer.segment(pages_meta, toc_analyses=toc_analyses)

    print("\nSegments found:", len(segments))
    for seg in segments[:30]:
        print(
            f"pages {seg.start_page:>4}-{seg.end_page:<4} "
            f"({seg.page_count:>4}p) source={seg.source:<12} "
            f"confidence={seg.confidence:.2f}  {seg.title or ''}"
        )
else:
    print("Set STRUCTURE_METHOD_NAME after inspecting the available methods.")


Method: segment
Signature: (pages: 'list[PageRepresentation]', toc_analyses: 'list[TOCAnalysis] | None' = None, title_candidates: 'dict[int, list[TitleCandidate]] | None' = None) -> 'list[DocumentSegment]'

Segments found: 69
pages    0-0    (   1p) source=toc          confidence=0.90  Mechanical Part Support & Fan Guard
pages    1-1    (   1p) source=toc          confidence=0.90  Introduction…………………...................................
pages    2-2    (   1p) source=toc          confidence=0.90  Namepalte Drawing
pages    3-3    (   1p) source=toc          confidence=0.90  INTRODUCTION ................................................................................................................................................
pages    4-4    (   1p) source=toc          confidence=0.90  INSTRUCTIONS POUR LE STOCKAGE ...................................................................................................................
pages    5-6    (   2p) source=toc          confidence=0

## 16. Bookmarks ↔ detected structure

Compare the PDF-native outline with printed TOCs and title candidates.

Agreement is strong evidence. Disagreement is not necessarily a failure: PDFs can contain incomplete, stale or missing bookmarks.


In [16]:
bookmark_df = pd.DataFrame([
    {"level": item[0], "title": item[1], "pdf_page": item[2]}
    for item in outline
])

display(bookmark_df.head(30))


""


## 17. Document boundaries

A TOC is **not automatically a document boundary**.

Boundary evidence can include:

- large/top-of-page title;
- document-type wording;
- cover-like layout;
- document identifier;
- manufacturer change;
- page-numbering change;
- major layout change;
- following TOC.

Nested/local TOCs must therefore be treated differently from a new embedded document.


## 18. Nested/local TOCs

A large PDF can have:

```text
global TOC
  ↓
section
  ↓
local TOC
  ↓
subsections
```

Therefore:

```text
TOC detected ≠ split document
```

The decision needs surrounding context and hierarchy.


## 19. Hierarchy and page ranges

The target is a structure such as:

```text
Document
├── 1 Introduction
│   ├── 1.1 Scope
│   └── 1.2 Safety
├── 2 Installation
└── 3 Maintenance
```

Then infer page ranges while keeping:

- printed page references;
- PDF page indices;
- hierarchy level;
- provenance;
- confidence

as separate concepts.


## 20. Structure → downstream routing

This is the main reason for the entire structure layer.

Instead of:

```text
500 pages → expensive LLM / NER / OCR on every page
```

we want:

```text
500 pages
  ↓
cheap extraction + structural analysis
  ↓
document architecture
  ↓
useful sections/pages
  ↓
expensive extraction only where justified
```

The goal is **high useful-page recall with substantial computational savings**.


In [17]:
toc_pages = sorted({
    page
    for region in detected_tocs
    for page in region.page_numbers
})

print("Total PDF pages:", len(pages_meta))
print("Detected TOC pages:", toc_pages)
if pages_meta:
    print("TOC-page fraction:", f"{len(toc_pages) / len(pages_meta):.1%}")


Total PDF pages: 1068
Detected TOC pages: [5, 6, 188, 282, 283, 284, 285, 409, 549, 554, 599, 601, 603, 635, 636, 751, 753, 776, 794, 876, 938, 1000, 1068]
TOC-page fraction: 2.2%
